# Grid search

Compare free-space routing, obstacle inflation and blocked routes.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Map

This planner searches cell centers with eight neighbors and prevents cutting diagonal corners.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.planning.search import astar
from autonomous_vehicles.simulation.occupancy import inflate_obstacles

occupied = np.zeros((50, 80), dtype=bool)
occupied[5:35, 30:34] = True
occupied[25:45, 50:54] = True
start, goal = (10, 5), (40, 72)

## Shortest route

Costs are one per axial step and sqrt(2) per diagonal step, measured in cell units.

In [ ]:
result = astar(occupied, start, goal)
assert result.reached
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(occupied, origin="lower", cmap="gray_r")
ax.plot([cell[1] for cell in result.cells], [cell[0] for cell in result.cells], color="tab:blue")
ax.scatter([start[1], goal[1]], [start[0], goal[0]], marker="x", color="red")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.show()
print("Cost:", result.cost, "expanded:", result.expanded)

## Clearance sensitivity

Inflation reduces traversable space before planning.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, radius in zip(axes, (0, 2, 4), strict=True):
    expanded = inflate_obstacles(occupied, radius)
    path = astar(expanded, start, goal)
    ax.imshow(expanded, origin="lower", cmap="gray_r")
    if path.reached:
        ax.plot([c[1] for c in path.cells], [c[0] for c in path.cells])
    ax.set_title(f"Radius {radius}, reached={path.reached}")
plt.tight_layout()
plt.show()

## Blocked corridor

A failed search has no cost and no path. Callers must handle it explicitly.

In [ ]:
blocked = occupied.copy()
blocked[:, 40] = True
failure = astar(blocked, start, goal)
print(failure)
assert not failure.reached
assert failure.cost is None
assert failure.cells == ()

## Budget

A search budget provides deterministic termination but does not produce an optimal partial route.

In [ ]:
for budget in (1, 20, 200, 5000):
    trial = astar(occupied, start, goal, maximum_expansions=budget)
    print(
        {"budget": budget, "expanded": trial.expanded, "reached": trial.reached, "cost": trial.cost}
    )
np.testing.assert_array_equal(occupied[5:35, 30:34], True)

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.